# 01 — Global Preprocessing

**Objective.** Apply the treatments that are **identical for all three objectives** — the ones that
fix data-quality problems rather than serve a particular model — and write the single cleaned dataset
that Objectives 1, 2 and 3 all start from.

Every treatment here follows from the evidence gathered in the preliminary analysis, and each section
is preceded by the evidence that justifies it. Where the preliminary evidence is not enough to choose
between options, this notebook measures the options first and decides afterwards.

This notebook applies eight treatments to the raw data. Each section presents the evidence, measures
the options where there's a choice, and records what was decided and why.

**Input:** `data/raw/SupplyFlow FMCG Solutions.xlsx`
**Output:** `data/preprocessed/warehouse_preprocessed.csv`

### Deliberately *not* done here

Each of these depends on the algorithm, so it belongs to the objective that needs it: encoding
categorical columns · scaling · deriving new features · building the Objective 2 target · resampling
for class imbalance · handling the collinear block found in the preliminary analysis.

## 0. Setup

In [ ]:
import sys, pathlib

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "common.py").exists())
sys.path.insert(0, str(ROOT))
from src.common import *

set_style()

raw = load_raw()      # kept untouched, for before/after comparisons
df = raw.copy()       # every treatment is applied to this copy

print(f"loaded {df.shape[0]:,} rows x {df.shape[1]} columns from {RAW_FILE.name}")

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
print(f"Ware_house_ID distinct values : {df['Ware_house_ID'].nunique()} (rows: {len(df)})")
print(f"WH_Manager_ID distinct values : {df['WH_Manager_ID'].nunique()}")

---
## 1. Identifier columns

**Evidence.**

The cell above confirms that both `Ware_house_ID` and `WH_Manager_ID` have exactly as many distinct
values as there are rows — every identifier is unique. No duplicate records were found in the
preliminary analysis. Neither identifier describes how a warehouse operates; a tree-based model
could split on row identity and fit the training data without learning anything useful.

**Decision flow.**

- **Do not use either ID as a feature.** They identify rows; they do not explain operations.
- **Keep `Ware_house_ID` as the row key.** Objective 3 gives a shipment-weight recommendation for each warehouse, so every output must be traceable to the warehouse it describes.
- **Drop `WH_Manager_ID`.** It identifies the same rows as `Ware_house_ID`, adds no analytical information, and is personal data.

> **Decision — identifier columns.**
>
> - Keep `Ware_house_ID` in every file as the row key.
> - Never use `Ware_house_ID` as a model feature.
> - Drop `WH_Manager_ID`.

In [ ]:
assert df["Ware_house_ID"].is_unique, "Ware_house_ID must stay unique to serve as the key"

df = df.drop(columns=["WH_Manager_ID"])

print("dropped         : WH_Manager_ID")
print(f"kept as row key : Ware_house_ID  (still unique: {df['Ware_house_ID'].is_unique})")
print(f"shape now       : {df.shape[0]:,} rows x {df.shape[1]} columns")

> **Interpretation.**
>
> - One column removed, 23 remain, and all 25,000 rows are intact. The key is still
>   unique, so every later result — a cluster label, a risk class, a predicted shipment weight — can be
>   joined back to the warehouse it describes.


In [ ]:
cert = "approved_wh_govt_certificate"
no_cert = df[cert].isna()
print(f"Missing certificates (literal 'NA' in file): {int(no_cert.sum())} rows")
print(f"\nMean values — unrated vs rated warehouses:")
for col in ["wh_breakdown_l3m", "storage_issue_reported_l3m", "product_wg_ton"]:
    u = df.loc[no_cert, col].mean()
    r = df.loc[~no_cert, col].mean()
    print(f"  {col}: {u:.2f} (unrated)  vs  {r:.2f} (rated)")

set_nc  = set(df.index[no_cert])
set_zbd = set(df.index[df["wh_breakdown_l3m"] == 0])
set_zsi = set(df.index[df["storage_issue_reported_l3m"] == 0])
j_bd = len(set_nc & set_zbd) / len(set_nc | set_zbd)
j_si = len(set_nc & set_zsi) / len(set_nc | set_zsi)
print(f"\nJaccard(no-cert, zero-breakdown): {j_bd:.4f}")
print(f"Jaccard(no-cert, zero-storage):   {j_si:.4f}")
print(f"\nMean est. year: unrated {df.loc[no_cert, 'wh_est_year'].mean():.2f}"
      f"  vs rated {df.loc[~no_cert, 'wh_est_year'].mean():.2f}")

---
## 2. The certificate's `'NA'` and a flag

**Evidence.**

The cell above confirms the key numbers from the preliminary analysis. All 908 null certificates are
cells containing the literal text `'NA'` — not blanks. The group differs from rated warehouses by
two to three standard deviations on breakdowns, storage issues and shipment. The Jaccard coefficient
between the certificate-missing rows and the zero-storage and zero-breakdown rows is exactly 1.0000,
meaning all three conditions select the same 908 warehouses. They were established on average thirteen
years later than the rest of the network.

**Options.**

| Option | What it would do | Verdict |
|---|---|---|
| Fill with the most common grade | `C` is the most common grade at 22.00%. Filling would issue 908 warehouses a certificate the file records they do not hold. | rejected — invents data |
| Drop the 908 rows | Removes the whole recently-commissioned segment | rejected — contradicts the 25,000-warehouse scope |
| Keep as a level of its own | Preserves exactly what the file records | **adopted** |

**Decision flow.**

- **Treat `'NA'` as a recorded state.** It marks a distinct warehouse subgroup, not a missing certificate grade.
- **Add `Unrated` as its own certificate level.** This keeps the source meaning visible.
- **Do not place `Unrated` on the C < B < B+ < A < A+ grade scale.** Putting it below C would claim the unrated warehouses are worse than grade C, even though they log no storage issues or breakdowns.
- **Add a separate 0/1 flag.** Each objective can then encode the five certificate grades in order and carry the unrated state separately.
- **Define the flag from the certificate, not from breakdowns.** A flag built from the Objective 2 target would carry the answer into the features.

> **Decision — certificate treatment.**
>
> - Replace the certificate's `'NA'` with the level **`Unrated`**.

> **Decision — unrated flag.**
>
> - Add **`is_unrated_warehouse`** where 1 means unrated.
> - Define it from `approved_wh_govt_certificate`, a recorded warehouse attribute.
> - Do not define it from `wh_breakdown_l3m`.

In [ ]:
cert = "approved_wh_govt_certificate"

# NB 00 §4 showed every null in this column is the literal text 'NA' in the file,
# so this fill touches exactly those cells and nothing else.
n_filled = int(df[cert].isna().sum())
df[cert] = df[cert].fillna("Unrated")
df["is_unrated_warehouse"] = (df[cert] == "Unrated").astype("int64")

print(f"cells filled with 'Unrated' : {n_filled}")
print(f"is_unrated_warehouse = 1    : {int(df['is_unrated_warehouse'].sum())}\n")
print(df[cert].value_counts().to_string())

> **Interpretation.**
>
> The value_counts output shows six certificate levels in the treated column: the original five grades
> (C, B+, B, A, A+) with their counts unchanged, and `Unrated` at exactly 908 rows. All 908 affected
> cells were relabelled and none were left as nulls.

In [ ]:
# The flag should select exactly the rows NB 00 §10 identified. Re-checked here on the
# treated data, so the claim does not rest on the earlier notebook alone.
pd.crosstab(df["is_unrated_warehouse"],
            [df["storage_issue_reported_l3m"].eq(0).rename("storage_issue == 0"),
             df["wh_breakdown_l3m"].eq(0).rename("breakdown == 0")])

> **Interpretation.**
>
> - Exactly 908 cells were filled, and the certificate now has six levels: the five
>   grades with their counts unchanged (C 5,501 down to A+ 4,191) plus `Unrated` at 908.
>
> - The cross-tabulation confirms the flag on the treated data. All 908 flagged warehouses have zero
>   storage issues *and* zero breakdowns; all 24,092 unflagged warehouses have neither; and the two
>   off-diagonal cells are empty.
>
> - **A caution for Objective 2, recorded now so it is not rediscovered later.** Because the flag
>   coincides exactly with `wh_breakdown_l3m == 0`, any model given this flag — or the `Unrated` level,
>   or `storage_issue_reported_l3m` — can identify those 908 warehouses as zero-breakdown with certainty.
>   That is not leakage in the strict sense: all three are recorded attributes, not the target. But it
>   means part of any Objective 2 score will come from an easy subgroup. Under the binary banding these
>   908 sit inside **Not High Risk** (13,026 warehouses), where they make up 6.97% of the class —
>   present, but not large enough to flatter the result on their own.


---
## 3. `workers_num`

**Evidence.**

From the preliminary analysis:
- 990 gaps (3.96%), true blanks, spread evenly through the file.
- The gap is unrelated to either target, but rows missing a worker count are more often
  flood-impacted (0.23 vs 0.09, SMD +0.391), flood-proof (0.16 vs 0.05, +0.368) and electric-backed
  (0.74 vs 0.65, +0.190).
- Right-skewed: skew 1.060, median 28, Q3 33, maximum 98.
- `workers_num` correlates with `electric_supply` (0.340) and `flood_impacted` (0.168).

**Dropping the 990 rows is not an option worth measuring.** The gap is unrelated to either target, so
removal would cost 3.96% of the data for nothing — and the rows removed would over-represent
flood-exposed sites.

**Mean or median?** With skew 1.060 and a maximum of 98 against a median of 28, a small number of very
large warehouses pull the mean upward. The median is not affected by them. Both are printed below.

**One value for everyone, or one per group?** The rows being filled are more often flood-impacted,
flood-proof and electric-backed, and the preliminary analysis shows worker counts vary with two of those.
*If* warehouses in those groups typically employ different numbers of workers, a single overall median
would misstate the filled values in a consistent direction. The table measures how far apart the group
medians really are.

In [ ]:
w = "workers_num"
known = df[w].notna()

print(f"all warehouses with a recorded count: median {df[w].median():.1f}, mean {df[w].mean():.2f}\n")

rows = []
for indicator in ["electric_supply", "flood_impacted", "flood_proof"]:
    for level in [0, 1]:
        in_level = df[indicator] == level
        rows.append({
            "indicator": indicator,
            "level": level,
            "warehouses_with_count": int((in_level & known).sum()),
            "gaps_to_fill": int((in_level & ~known).sum()),
            "median_workers": df.loc[in_level, w].median(),
            "mean_workers": round(df.loc[in_level, w].mean(), 2),
        })
pd.DataFrame(rows)

> **Interpretation.**
>
> - The group medians are not all the same, and one indicator separates them clearly.
>   - **Electric back-up:** median **24** workers without it, **30** with it — six workers apart, either side of the overall 28.
>   - **Flood impact:** 28 when not flood-impacted, 32 when flood-impacted — four apart.
>   - **Flood-proofing:** 28 without, 30 with — two apart.
>
> - In every group the mean sits above the median (overall 28.94 against 28.0), as the right skew predicts.
>
> - Every one of the 990 gaps falls into one of the two electric-supply groups:
>   - 257 warehouses without back-up;
>   - 733 warehouses with back-up.
>
> - Filling all gaps with the overall median of 28 would misstate both groups:
>   - no-backup warehouses would receive four workers more than their group median;
>   - backed-up warehouses would receive two fewer than their group median.

> **Decision — workers_num imputation.**
>
> - Fill `workers_num` with the **median of its `electric_supply` group**.
> - Use the **median, not the mean**, because `workers_num` is right-skewed (skew 1.060).
> - Group by **electric supply**, because it creates the largest median shift and both groups are large enough to be stable.
> - Do not group by all three indicators; that would create up to eight groups for a small correction on a few hundred rows.
> - Do not add a missing-indicator flag; the gap is unrelated to both targets, and the columns it does relate to are already present.

**Is it safe to compute the fill values before the train/test split?**

The project follows a leakage rule: anything learned from the data — an imputation value included — is
fitted on the **training split only**, so nothing about the held-out test rows can influence training.
This notebook runs *before* any split exists, so the rule has to be checked, not assumed.

The helper below recomputes a fill value on **200 random subsets of 80% of the rows** — the size of a
training split — and counts how many reproduce the value obtained from all 25,000 rows. If every
subset reproduces it exactly, then filling here produces exactly the same data as filling after the
split would, and no information can leak. If any subset differs, the fill has to move into each
objective's own notebooks, after its split.

In [ ]:
# Leakage check: does a random 80% training subset reproduce the fill values exactly?
# If every subset reproduces them, filling here gives identical data to filling after a split.
full_value = df.groupby("electric_supply")["workers_num"].median().to_dict()
rng = np.random.default_rng(RANDOM_STATE)
n_rows = int(0.8 * len(df))
matches = sum(
    df.iloc[rng.choice(len(df), n_rows, replace=False)]
      .groupby("electric_supply")["workers_num"].median().to_dict() == full_value
    for _ in range(200)
)
worker_fill = full_value

print(f"fill values from all 25,000 rows          : {worker_fill}")
print(f"random 80% subsets reproducing them exactly: {matches} of 200")

> **Interpretation.**
>
> - All **200 of 200** random training-sized subsets reproduce the fill values
>   exactly: 24 workers without electric back-up, 30 with. A median over thousands of whole-number
>   values per group is extremely stable — removing a random fifth of the rows does not move it.
>   Filling here therefore produces exactly the data that filling after each objective's split would,
>   so the leakage rule is satisfied in substance rather than assumed.


In [ ]:
before_w = df[w].copy()
df[w] = df[w].fillna(df["electric_supply"].map(worker_fill))

print(f"gaps before: {int(before_w.isna().sum())}   gaps after: {int(df[w].isna().sum())}\n")
print("values given to the filled rows, by electric_supply group:")
print(pd.crosstab(df.loc[before_w.isna(), "electric_supply"],
                  df.loc[before_w.isna(), w]).to_string())

> **Interpretation.**
>
> The filling worked as decided: the 257 warehouses without electric back-up received 24 workers and
> the 733 with back-up received 30. No gaps remain in `workers_num` after this step.

In [ ]:
pd.DataFrame({
    "recorded counts only": before_w.describe(),
    "after filling": df[w].describe(),
}).round(3)

> **Interpretation.**
>
> - All 990 gaps are filled, exactly as decided: the 257 warehouses without back-up
>   received 24, and the 733 with back-up received 30. The column barely changes — mean 28.944 → 28.925,
>   standard deviation 7.873 → 7.733 — and its minimum, quartiles (24, 28, 33) and maximum are
>   untouched. Under 4% of values were filled, at values close to the centre, so the treatment keeps
>   990 rows usable without distorting what the recorded counts say.


---
## 4. `wh_est_year`

**Evidence.**

From the preliminary analysis:
- 11,881 gaps (**47.52%**), true blanks, spread evenly through the file.
- Flat across every categorical, but rows without a year have a distinct operating profile:
  refill requests 2.55 vs 5.49 (SMD −1.353), transport issues 1.13 vs 0.45 (+0.589), temperature
  regulation 0.20 vs 0.40 (−0.451), shipment 20,100.85 vs 23,915.51 t (−0.334), storage issues 15.80
  vs 18.33 (−0.279).
- Among warehouses that have a year, it correlates with storage issues (−0.859), shipment
  weight (−0.829) and breakdowns (−0.399). It is the breakdown target's strongest correlate, and
  warehouse age is one of the two business drivers being tested.

That last point rules out **dropping the column**: it would discard the strongest single correlate of
breakdowns and one of the two named drivers.

Three options remain, and each can be measured instead of argued:

- **(a) Drop the rows with no year** — keeps every recorded value exactly, at the cost of rows.
- **(c) Fill with the median year** — simple; every filled row receives the same value.
- **(d) Fill from storage issues** — fit a straight line of year on `storage_issue_reported_l3m`, its
  strongest correlate, using warehouses that have a year, then predict the missing ones.

For each, the cell below measures what it does to the rows kept and to the year's relationship with
the three columns it relates to most. The recorded years are the reference: a good treatment should
leave those relationships close to where the recorded data puts them.

In [ ]:
y = "wh_est_year"
has_year = df[y].notna()
related = ["storage_issue_reported_l3m", "product_wg_ton", "wh_breakdown_l3m"]

# (c) median
median_year = df[y].median()
fill_median = df[y].fillna(median_year)

# (d) straight line of year on storage issues, fitted only on warehouses that have a year
slope, intercept = np.polyfit(df.loc[has_year, "storage_issue_reported_l3m"], df.loc[has_year, y], 1)
predicted = intercept + slope * df.loc[~has_year, "storage_issue_reported_l3m"]
fill_line = df[y].copy()
fill_line[~has_year] = predicted

options = pd.DataFrame([
    {"option": "reference: recorded years only", "rows": int(has_year.sum()),
     **{f"r_with_{c}": round(df.loc[has_year, y].corr(df.loc[has_year, c]), 3) for c in related}},
    {"option": "(c) fill with median year",      "rows": len(df),
     **{f"r_with_{c}": round(fill_median.corr(df[c]), 3) for c in related}},
    {"option": "(d) fill from storage issues",   "rows": len(df),
     **{f"r_with_{c}": round(fill_line.corr(df[c]), 3) for c in related}},
])

lo, hi = df[y].min(), df[y].max()
print(f"(a) dropping rows keeps {int(has_year.sum()):,} of {len(df):,} ({100 * has_year.mean():.2f}%);"
      f" its relationships are the reference row below")
print(f"    mean shipment of the rows kept: {df.loc[has_year, 'product_wg_ton'].mean():,.0f} t"
      f"   vs all rows: {df['product_wg_ton'].mean():,.0f} t\n")
print(f"(c) median year {median_year:.0f}; after filling, rows at exactly {median_year:.0f}: "
      f"{100 * (fill_median == median_year).mean():.2f}% of all rows\n")
print(f"(d) fitted line: year = {intercept:.2f} {slope:+.4f} x storage_issue_reported_l3m")
print(f"    predicted years for the gaps run from {predicted.min():.1f} to {predicted.max():.1f};"
      f" recorded years run {lo:.0f} to {hi:.0f}")
print(f"    predictions outside the recorded range: {int(((predicted < lo) | (predicted > hi)).sum()):,}\n")
options

> **Interpretation.**
>
> The options table shows that no approach preserves the year's relationships unchanged. Dropping rows
> keeps only 52.48% of the network and biases the retained sample toward heavier-shipping warehouses
> (mean 23,916 t vs 22,103 t for all warehouses). Median filling weakens the year's correlations with
> storage issues and shipment substantially. Prediction from storage issues strengthens those
> correlations beyond anything the recorded data shows and produces years outside the recorded range.

In [ ]:
# Would a median taken from *similar* warehouses give a different year?
# NB 00 §5: the rows missing a year differ most on these three columns.
for c in ["num_refill_req_l3m", "transport_issue_l1y", "temp_reg_mach"]:
    by_level = df.loc[has_year].groupby(c)[y].median()
    print(f"median recorded year by {c:<20}: "
          + "  ".join(f"{k}->{v:.0f}" for k, v in by_level.items()))

> **Interpretation.**
>
> Among warehouses with a recorded year, the median is 2009 or 2010 at every refill level from 3 to 8,
> at every transport-issue level, and at both temperature-regulation levels. A group median would
> produce the same fill value as the overall median in every case, so grouping adds no benefit here.

In [ ]:
# The medians above list no refill level below 3 and no transport-issue level of 5, which
# would mean some levels have no recorded year at all. Rather than infer that from levels
# absent from a printout, every column with few distinct values is checked directly for
# levels where the year is missing for ALL warehouses or for NONE. Levels with fewer than
# 30 warehouses are skipped: a tiny group can be all-or-nothing by chance.
gap = df[y].isna()
low_cardinality = [c for c in df.columns
                   if c not in (y, "Ware_house_ID") and df[c].nunique() <= 45]

all_or_nothing = []
for c in low_cardinality:
    for level, g in gap.groupby(df[c]):
        if len(g) >= 30 and g.mean() in (0.0, 1.0):
            all_or_nothing.append({"column": c, "level": level, "warehouses": len(g),
                                   "pct_no_year": round(100 * g.mean(), 2)})

print(f"columns scanned: {len(low_cardinality)}")
print("levels where the year is missing for every warehouse, or for none:")
print(pd.DataFrame(all_or_nothing).to_string(index=False) if all_or_nothing else "  none")

print("\nfull breakdown, one row per level, for each column listed above:")
for c in sorted({r["column"] for r in all_or_nothing}):
    table = pd.DataFrame({
        "warehouses": df.groupby(c).size(),
        "no_recorded_year": gap.groupby(df[c]).sum(),
        "pct_no_year": (100 * gap.groupby(df[c]).mean()).round(2),
    })
    print(f"\n--- {c} ---")
    print(table.to_string())

> **Interpretation.**
>
> - **Every option distorts something; they differ in the direction.**
>
> | Option | What the measurement shows |
> |---|---|
> | (a) drop rows | keeps 13,119 of 25,000 (52.48%). Mean shipment of the rows kept is 23,916 t against 22,103 t for all warehouses — the network would look about 1,800 t heavier than it is. |
> | (c) median | the year's relationships **weaken**: with storage issues −0.859 → −0.629, with shipment −0.829 → −0.605, with breakdowns −0.399 → −0.288. And 49.46% of all rows end up at exactly 2009. |
> | (d) line from storage issues | the relationships **strengthen beyond anything recorded**: with storage issues −0.859 → −0.915, with shipment −0.829 → −0.894 (with breakdowns −0.399 → −0.380). And 107 predicted years fall outside the recorded range — predictions reach 1995.1, before the earliest recorded year of 1996. |
>
> - **A median from similar warehouses would change nothing.** At every refill level from 3 to 8, at every transport-issue level recorded, and at both temperature-regulation levels, the median recorded year is 2009 or 2010.
>
> - **The scan explains the gap pattern.** Of the 17 columns scanned, exactly two contain levels where the year is missing for every warehouse:
>   - **refill requests 0, 1 and 2:** year missing for all 7,576 warehouses;
>   - **transport issues = 5:** year missing for all 348 warehouses.
>
> - So the gap in `wh_est_year` is not random:
>   - whether a year is recorded follows values in other columns;
>   - among warehouses with 0–2 refill requests there is no recorded year from which to take a group median;
>   - the missingness itself carries information that later models should be able to see.

> **Decision — wh_est_year imputation.**
>
> - Fill `wh_est_year` with the **overall median, 2009**.
> - Add **`wh_est_year_missing`** where 1 means the year was filled.
> - Reject dropping rows: it removes 47.52% of the network and removes whole operating segments.
> - Reject prediction from storage issues: it creates relationships stronger than the recorded data and produces 107 years outside the recorded range.
> - Accept median filling with its cost stated: it weakens the year's relationships, which is safer than inventing a stronger one.
> - Keep the flag: a recorded 2009 and a filled 2009 should remain distinguishable.

In [ ]:
# Same leakage check for the year median.
full_value = df["wh_est_year"].median()
rng = np.random.default_rng(RANDOM_STATE)
n_rows = int(0.8 * len(df))
matches = sum(
    df.iloc[rng.choice(len(df), n_rows, replace=False)]["wh_est_year"].median() == full_value
    for _ in range(200)
)
year_fill = full_value

print(f"median year from all 25,000 rows         : {year_fill}")
print(f"random 80% subsets reproducing it exactly: {matches} of 200")

In [ ]:
before_y = df[y].copy()

df["wh_est_year_missing"] = df[y].isna().astype("int64")    # recorded BEFORE filling
df[y] = df[y].fillna(year_fill)

print(f"wh_est_year_missing = 1 : {int(df['wh_est_year_missing'].sum()):,}")
print(f"gaps before / after     : {int(before_y.isna().sum()):,} / {int(df[y].isna().sum())}\n")
pd.DataFrame({
    "recorded years only": before_y.describe(),
    "after filling": df[y].describe(),
}).round(3)

> **Interpretation.**
>
> - The median year is reproduced by **200 of 200** random training-sized subsets, so
>   — as with `workers_num` — filling here is identical to filling after a split. The flag marks
>   exactly 11,881 warehouses, the number of gaps, and none remain.
>
> - **What filling did to the column must stay in view downstream.** The mean barely moves
>   (2,009.383 → 2,009.201), but the spread collapses: the standard deviation falls from 7.528 to 5.457,
>   and the interquartile range narrows from 2003–2016 to 2009–2010, because nearly half the rows now sit
>   at one value.
>
> - The filled column is fit for **modelling alongside its flag**. It is **not** fit for **describing
>   the age profile of the network**. Any distribution, average or plot of establishment year or
>   warehouse age in a later notebook must use `wh_est_year_missing == 0` rows only — and the same holds
>   for any warehouse-age feature derived from this column, which would inherit the spike at 2009.


---
## 5. Integer types

**Evidence.**

From the preliminary analysis:
- `workers_num` was stored as `float64` only because it contained gaps.
- `wh_est_year` was stored as `float64` only because it contained gaps.
- Both columns are whole-number fields by definition: a count of people and a calendar year.
- Both columns are now gap-free.
- The conversion cell checks every value before converting, so no decimal value can be silently truncated.

> **Decision — integer types.**
>
> - Convert `workers_num` to `int64`.
> - Convert `wh_est_year` to `int64`.
> - Convert only after the whole-number check passes.

In [ ]:
for c in ["workers_num", "wh_est_year"]:
    assert df[c].notna().all(), f"{c} still has gaps"
    assert (df[c] % 1 == 0).all(), f"{c} has non-whole values — converting would truncate them"
    df[c] = df[c].astype("int64")

print(df[["workers_num", "wh_est_year"]].dtypes.to_string())
print("\nall dtypes:", df.dtypes.astype(str).value_counts().to_dict())

> **Interpretation.**
>
> - Both columns passed the whole-number check before conversion, so nothing was
>   truncated. The dataset now holds **18 integer and 7 text columns and no floats** — every numeric
>   value in the file is a whole number, which matches what every column measures.


---
## 6. Outliers

**Evidence.**

From the preliminary analysis:
- The three highest IQR flag rates — `transport_issue_l1y` (11.77%), `flood_impacted` (9.82%) and
  `flood_proof` (5.46%) — are artefacts of applying a continuous-variable rule to a count and two 0/1
  indicators. For the indicators the fence collapses to `[0, 0]` and every 1 is flagged.
- `Competitor_in_mkt` flags 96 rows (0.38%), all above 7, maximum 12.
- The only continuous columns with meaningful flag rates are `retail_shop_num` (81 below 2,532.5 and
  867 above 7,280.5) and `workers_num` (5 below 10.5 and 602 above 46.5).
- Ten of sixteen numeric columns flag nothing, both targets included. No negative or otherwise impossible values were found anywhere.

Before deciding on the two continuous columns, the cell below looks at the extreme rows themselves and
asks one plausibility question the preliminary analysis did not: does a warehouse's worker count follow its capacity size?
If it did, a very high count in a Small warehouse would be suspicious.

In [ ]:
print("median workers_num by WH_capacity_size (recorded counts only):")
print(raw.groupby("WH_capacity_size")["workers_num"].median().to_string())

context = ["Ware_house_ID", "WH_capacity_size", "Location_type", "zone",
           "workers_num", "retail_shop_num", "distributor_num", "product_wg_ton"]
for c in ["retail_shop_num", "workers_num"]:
    s = df.loc[raw[c].notna()].sort_values(c)       # recorded values only
    print(f"\n=== {c}: five lowest and five highest ===")
    print(pd.concat([s.head(5), s.tail(5)])[context].to_string(index=False))

> **Interpretation.**
>
> The extreme-row table shows that both the lowest and highest worker counts appear across all three
> capacity sizes and in all zones, with ordinary shipping volumes. Worker count does not follow
> warehouse size: the median is 28 for Large, Mid and Small alike, so a high count in a Small
> warehouse is not contradicted by the data.

In [ ]:
# All five of the highest worker counts above are exactly 98. A single extreme value that
# repeats can be a cap or a code rather than a real count, so the upper tail is listed in full.
upper = raw.loc[raw["workers_num"] > 60, "workers_num"].value_counts().sort_index()
print(f"warehouses with more than 60 recorded workers: {int(upper.sum())}\n")
print(upper.to_string())

> **Interpretation.**
>
> - **Worker count does not follow capacity size.** The median is 28 in Large, Mid and Small warehouses
>   alike. So a high worker count in a Small warehouse is not contradicted by anything else in the data —
>   and capacity cannot be used to judge whether an extreme count is wrong.
>
> - **The extreme `retail_shop_num` rows are unremarkable.** The five lowest (1,821 to 1,953) and five
>   highest (10,224 to 11,008) span all three capacity sizes and two zones, are all Rural, and have
>   ordinary worker and distributor counts, with shipments from 6,106 to 51,124 t. Nothing in them marks
>   them as errors.
>
> - **The upper tail of `workers_num` has a pattern real staffing records would be unlikely to produce.**
>   59 warehouses record more than 60 workers. The value 61 occurs 14 times — but **every value above it
>   (62, 63, 64, 65, 67, 72, 78, 92 and 98) occurs exactly five times**: nine values, 45 warehouses. The
>   minimum, 10, also occurs exactly five times. These rows are not duplicates, and their other attributes
>   are ordinary. The regularity suggests these extremes may have been constructed rather than recorded,
>   but nothing in the file shows which values are wrong, or what they should be instead.

> **Decision — outliers: retain every value.**
>
> - Nothing is capped.
> - Nothing is removed.
>
> - `transport_issue_l1y`, `flood_impacted`, `flood_proof` — their flags are artefacts of the IQR rule
>   applied to a count and two 0/1 indicators. Acting would delete real minority levels.
> - `Competitor_in_mkt`, `retail_shop_num` — the extremes are possible values, and the extreme rows are
>   unremarkable on every other column.
> - `workers_num` — retained, with the regular pattern recorded as a **known anomaly**. Capping would
>   replace 50 recorded values (0.20% of the network) with invented ones on the strength of a suspicion.
> - This global decision is deliberately minimal. Methods that are sensitive to extreme values —
>   K-Means distances in Objective 1, least-squares fits in Objective 3 — check that sensitivity in
>   their own transformation notebooks, where the right handling depends on the algorithm.

In [ ]:
# The outlier decision changes nothing. Confirm it: every recorded value in the flagged columns is
# identical to the raw file.
flagged = ["transport_issue_l1y", "flood_impacted", "flood_proof",
           "Competitor_in_mkt", "retail_shop_num", "workers_num"]
for c in flagged:
    recorded = raw[c].notna()
    assert (df.loc[recorded, c] == raw.loc[recorded, c]).all(), f"{c} was altered"
print("recorded values unchanged in:", ", ".join(flagged))

---
## 7. Rows

**Evidence.**

- The preliminary analysis found no duplicate records.
- The cross-column analysis showed the 908 unusual warehouses are a coherent subgroup, not erroneous rows.
- The project scope keeps all 25,000 warehouses.
- Sections 1–6 filled gaps and added flags but removed nothing.

> **Decision — rows.**
>
> - Remove **no rows**.
> - Carry all 25,000 warehouses forward.
> - Preserve the original row order.

In [ ]:
assert len(df) == len(raw) == 25_000
assert df["Ware_house_ID"].equals(raw["Ware_house_ID"])
print(f"rows: {len(df):,} — the same warehouses, in the same order, as the raw file")

---
## 8. Save

In [ ]:
save_table(df, PREPROCESSED_FILE, index=False)

print("\ncolumns written:")
for i, c in enumerate(df.columns, 1):
    marker = "   <- added in this notebook" if c in ("is_unrated_warehouse", "wh_est_year_missing") else ""
    print(f"  {i:>2}. {c:<30} {str(df[c].dtype):<8}{marker}")

---
## 9. Checks

The file is read back from disk and compared with what was written, so the checks describe the file
the three objectives will actually load — not the dataframe in memory.

In [ ]:
back = load_preprocessed()

assert back.shape == (25_000, 25), back.shape
assert back.isna().sum().sum() == 0
assert back["Ware_house_ID"].is_unique
assert "WH_Manager_ID" not in back.columns
assert set(back["approved_wh_govt_certificate"]) == {"A+", "A", "B+", "B", "C", "Unrated"}
assert back["is_unrated_warehouse"].sum() == 908
assert back["wh_est_year_missing"].sum() == 11_881
pd.testing.assert_frame_equal(back, df.reset_index(drop=True))

print("all checks passed\n")
print(f"shape  : {back.shape[0]:,} rows x {back.shape[1]} columns")
print(f"nulls  : {int(back.isna().sum().sum())}")
print(f"dtypes : {back.dtypes.astype(str).value_counts().to_dict()}")

---
## Summary

**Output.** `data/preprocessed/warehouse_preprocessed.csv` — **25,000 rows × 25 columns**, no gaps,
18 integer and 7 text columns. It was read back from disk and verified identical to what was written.

**What this notebook did.** Eight treatments were applied in order. First, both identifier columns
were examined: `Ware_house_ID` was kept as the row key and `WH_Manager_ID` was dropped, since both
are one per row and neither describes warehouse operations. Next, the 908 literal `'NA'` entries in
the certificate column were relabelled as `Unrated` — a distinct level, not a gap — and a companion
binary flag was added so the objective notebooks can treat the five grade levels and the unrated
state separately. The `workers_num` gap (990 rows, 3.96%) was filled with the median for each
electric-supply group, because the group medians sit six workers apart; the overall median would
have understated every filled value in one group and overstated it in the other. The `wh_est_year`
gap (11,881 rows, 47.52%) was filled with the overall median of 2009, with a companion flag marking
which rows were filled; predicting from storage issues was rejected because it produced correlations
stronger than anything the recorded data shows and 107 years outside the observed range. After
filling, both columns were converted from `float64` to `int64`, since whole-number fields should not
carry a decimal dtype. Every outlier-flagged value was retained: the three highest IQR flag rates
are artefacts of applying a continuous rule to count and binary columns, and the genuine continuous
extremes are plausible. No rows were removed.

**New in this notebook that was not in the preliminary analysis.** Whether `wh_est_year` is recorded
turns out to follow other columns by an absolute rule: it is missing for every warehouse with 0–2
refill requests (7,576 warehouses) and for every warehouse with five transport issues (348). Both fill
values — 24 and 30 for workers, 2009 for year — are reproduced by all 200 of 200 random 80% subsets,
so filling here is identical to filling after a train/test split. Worker count does not vary with
capacity size; the median is 28 for Large, Mid and Small alike. And every `workers_num` value above 61
occurs exactly five times, as does the minimum of 10 — a regularity noted as a known anomaly.

**Cautions for the objective notebooks.** `Ware_house_ID` is the row key and must never be used as
a model feature. Any description or plot of warehouse age should use only the rows where
`wh_est_year_missing == 0`, because the fill put nearly half the rows at exactly 2009 and the
standard deviation of the column fell from 7.5 to 5.5. The filled year also understates the column's
relationships — for example its correlation with storage issues drops from −0.859 to −0.629. Because
`is_unrated_warehouse`, the `Unrated` certificate level and `storage_issue == 0` identify the same
908 zero-breakdown warehouses with certainty, Objective 2 should report performance with and without
them so the easy subgroup does not flatter the result. The 908 low-volume warehouses will weigh
heavily on MAPE in Objective 3. Extreme values were retained, including the regular upper tail in
`workers_num`, so scaling choices in the objective notebooks should reflect that.